In [1]:
import sys
sys.path.append('..')

import numpy as np
import torch
from tqdm import tqdm

%matplotlib inline
import matplotlib.pyplot as plt


In [2]:
def marginals(joint: torch.Tensor):
    # split joint samples into marginals X,Y
    dim = joint.shape[-1]
    mask = torch.zeros(dim, dtype=torch.bool, device=joint.device)
    mask[dim//2+1:] = True
    mask[1] = True
    return joint[:,~mask], joint[:,mask]


# exp config
device = torch.device('cuda:0')
hdgm_dim = 40
n_tests = 100
n_test_samples = 8000
n_subsets = 8
pretrained_path = "../exp/pretrained/deepkernel/hdgm40/mlp_small/best.pt"


In [3]:
from torch.utils.data import DataLoader
from data.toy import BlobHD
import metrics

# Define dataset / dataloader
# TODO: try using new dataset (direct sampling)
dataset = BlobHD(size=n_tests*n_test_samples,
                 dim=hdgm_dim,
                 var=1,
                 seed=0)
dataloader = DataLoader(dataset=dataset,
                        batch_size=n_test_samples,
                        shuffle=True,
                        drop_last=True,
                        num_workers=0,
                        pin_memory=True)


In [4]:
from model.mlp import MLP
from kernel import Gaussian, DeepKernel
from utils import utils

# Define model
featurizer = MLP(features=[20, 60, 60, 60],
                 activation='ReLU')             # NOTE: this should be ReLU for HDGM40 (even though config says ELU)
feature_kernel = Gaussian(trainable=True)
smoothing_kernel = Gaussian(trainable=True)
model = DeepKernel(featurizer, feature_kernel, smoothing_kernel, eps=0.01, trainable=True)
model = model.to(device)

# load model
utils.load_checkpoint(pretrained_path, model, device=device)
model.eval()
k = model
l = model

print('feature_kernel_bandwidth:', model.feature_kernel.bandwidth)
print('smoothing_kernel_bandwidth',model.smoothing_kernel.bandwidth)
print('deepkernel_eps:', model.eps)
print(model.featurizer.net[0].linear.weight)


feature_kernel_bandwidth: tensor([0.0735], device='cuda:0', grad_fn=<ExpBackward0>)
smoothing_kernel_bandwidth tensor([0.9991], device='cuda:0', grad_fn=<ExpBackward0>)
deepkernel_eps: tensor([0.0055], device='cuda:0', grad_fn=<SigmoidBackward0>)
Parameter containing:
tensor([[ 0.0483, -0.1649,  0.1435,  ..., -0.1851, -0.1606,  0.1628],
        [-0.2771, -0.2599,  0.1039,  ..., -0.0764, -0.3733, -0.1745],
        [-0.2195,  0.0120,  0.0827,  ...,  0.0981, -0.0477, -0.2105],
        ...,
        [-0.0927,  0.0748,  0.0264,  ..., -0.0822, -0.2385, -0.0799],
        [ 0.1472,  0.0804,  0.1924,  ..., -0.2864,  0.2064,  0.0181],
        [ 0.2512,  0.0405,  0.1862,  ..., -0.1200, -0.0284,  0.0078]],
       device='cuda:0', requires_grad=True)


In [5]:
with torch.no_grad():
    
    n_reject = 0
    n_reject_subtest = np.zeros(8)

    for batch in (pbar:=tqdm(dataloader)):
        joint = batch.to(device)
        X,Y = marginals(joint)
        hsic_8000, _, p_value_8000, _ = metrics.hsic.permutation_test(k,l,X,Y)
        if p_value_8000 < 0.05:
            n_reject += 1
        pbar.set_description(f"[p-value: {p_value_8000:.4f}]")

        subtest_sizes = [1000]*8
        chunk_idx = np.split(np.random.permutation(8000), np.cumsum(subtest_sizes))

        for i,idx in enumerate(chunk_idx[:-1]):
            hsic_1000, _, p_value_1000, _ = metrics.hsic.permutation_test(k,l,X[idx],Y[idx])
            if p_value_1000 < 0.05:
                n_reject_subtest[i] += 1


[p-value: 0.1740]: 100%|██████████| 100/100 [34:57<00:00, 20.97s/it]


In [6]:
power = n_reject/n_tests
power_subtests = n_reject_subtest/n_tests
print('power_8000:', power)
print('power_1000:', power_subtests)

power_8000: 0.16
power_1000: [0.07 0.05 0.09 0.07 0.08 0.04 0.05 0.06]


In [9]:
n_test_samples = 1000

dataset = BlobHD(size=n_tests*n_test_samples,
                 dim=hdgm_dim,
                 var=1,
                 seed=0)
dataloader = DataLoader(dataset=dataset,
                        batch_size=n_test_samples,
                        shuffle=True,
                        drop_last=True,
                        num_workers=0,
                        pin_memory=True)

with torch.no_grad():
    n_reject = 0
    for batch in (pbar:=tqdm(dataloader)):
        joint = batch.to(device)
        X,Y = marginals(joint)
        hsic_8000, _, p_value_8000, _ = metrics.hsic.permutation_test(k,l,X,Y)
        if p_value_8000 < 0.05:
            n_reject += 1
        pbar.set_description(f"[p-value: {p_value_8000:.4f}]")


[p-value: 0.5100]: 100%|██████████| 100/100 [00:28<00:00,  3.53it/s]


In [10]:
power = n_reject/n_tests
print('power_1000:', power)


power_1000: 0.56
